# Lab 1

In [ ]:
import passengersim as pax

In [ ]:
cfg = pax.Config.from_yaml(pax.demo_network("3MKT/DEMO"))

## Understanding the Impact of RM

We are going to run a series of experiments, so we'll set up an `Experiments`
class to keep track of and compare the results.

In [ ]:
from passengersim.experiments import Experiments

experiments = Experiments(cfg, output_dir="lab-1-output")

### First Come, First Served

First, let's suppose that neither carrier uses any RM optimization, and
just offers all fare classes as first come, first served, subject only 
to the existing restrictions (fare rules and advance purchase).  To 
simulate this, we'll create an RM system that does nothing and set the
config so that both carriers use this system.

In [ ]:
from passengersim.rm import RmAction, RmSys, register_rm_system


class Nothing(RmAction):
    """An RM step that does nothing."""

    frequency = "dcp"

    def run(self, sim: pax.Simulation, days_prior: int):
        return


@register_rm_system
class Z(RmSys):
    """A custom RM system that applies zero optimization."""

    availability_control = "leg"
    actions = [Nothing]

In [ ]:
@experiments
def ZZ(cfg: pax.Config) -> pax.Config:
    cfg.carriers.AL1.rm_system = "Z"
    cfg.carriers.AL2.rm_system = "Z"
    return cfg

In [ ]:
summary = experiments.run()

In [ ]:
summary.fig_carrier_revenues()

In [ ]:
summary.fig_carrier_load_factors()

In [ ]:
summary.fig_bookings_by_timeframe(by_carrier="AL1", by_class=True)

### AL1 implements EMSR-B

Now let's have one of the carriers implement EMSR-B, with EM demand untruncation and standard forecasting.

In [ ]:
@experiments
def EZ(cfg: pax.Config) -> pax.Config:
    cfg.carriers.AL1.rm_system = "E"
    cfg.carriers.AL2.rm_system = "Z"
    return cfg


summary = experiments.run()

In [ ]:
summary.fig_carrier_revenues()

In [ ]:
summary.fig_carrier_load_factors()

In [ ]:
summary.fig_bookings_by_timeframe(by_carrier="AL1", by_class=True, source_labels=True)

In [ ]:
summary.fig_bookings_by_timeframe(by_carrier="AL2", by_class=True, source_labels=True)

### Both Carriers implement EMSR-B

In [ ]:
@experiments
def EE(cfg: pax.Config) -> pax.Config:
    cfg.carriers.AL1.rm_system = "E"
    cfg.carriers.AL2.rm_system = "E"
    return cfg


summary = experiments.run()

In [ ]:
summary.fig_carrier_revenues()

In [ ]:
summary.fig_bookings_by_timeframe(by_carrier="AL1", by_class=True, source_labels=True)

## Understanding Relative Fares

For this section, we'll start a new collection of experiments.

In [ ]:
ex_fares = Experiments(cfg, output_dir="lab-1b-output")

We start with a baseline experiment where both carriers are using the "normal" fare structures and
the "E" RM system.

In [ ]:
@ex_fares
def Baseline(cfg: pax.Config) -> pax.Config:
    return cfg

In [ ]:
summary = ex_fares.run()
summary.fig_carrier_revenues()

In [ ]:
@ex_fares
def TopFareUp_OrdLax(cfg: pax.Config) -> pax.Config:
    for fare in cfg.fares:
        if fare.booking_class == "Y0" and fare.orig == "ORD" and fare.dest == "LAX":
            fare.price *= 1.1
    return cfg

In [ ]:
summary = ex_fares.run()
summary.fig_carrier_revenues()

In [ ]:
summary.fig_fare_class_mix()

In [ ]:
summary.fig_bookings_by_timeframe(by_carrier="AL1", by_class=True)

### Questions

- What happens if you mark up the price of the top fare in the BOS-ORD market instead?  Why is this different?
- What about BOS-LAX?  
- What if only AL1 makes this pricing change?